<div align="center">

# fse behind Kafka

Write vectors for a stream and continue at the right record after a crash.

</div>

## When do you need this?

A topic delivers records all day and never ends.
You want vectors on disk for every record, and after a crash the consumer must **continue at the right record**: no record lost, none stored twice.
fse can do this for a source that gives the **same records in the same order from a position on**. A Kafka partition does.

## How it works

- fse keeps, with every shard, the **position** of the batch in which the shard ends. A position is a text of your choice (up to 64 KiB). fse stores it and never reads it.
- After a restart, `sink.source_position` returns `(P, skip)`: **seek the consumer to `P`**, read again, and fse drops the first `skip` records, which the store holds already.
- `durable_rows` of each result says how many of the records read since the seek are safe on disk. **Commit to Kafka only up to there.**
- The store is the truth for a restart. The commit to Kafka only helps monitoring tools.

[Kafka guarantees](https://kafka.apache.org/41/getting-started/introduction/) that a consumer of a topic-partition reads the events of that partition in the order they were written. This recipe writes **one store for each partition**, because the stream-system gives no order across partitions.

## 1. A simulated partition

There is no broker here. A small class stands in for one partition: records have an **offset**, and the offsets have gaps, as they do after a compaction or a transaction marker.

In [ ]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work
encoder = recipe.encoder()
news = recipe.news(900)


class Partition:
    """One partition of a topic: records with an offset, a start (retention) and a commit."""

    def __init__(self, topic, number, texts):
        self.topic, self.number = topic, number
        self.offsets = [3 * i + 10 + (i % 2) for i in range(len(texts))]  # gaps between offsets
        self.texts = list(texts)
        self.start = self.offsets[0]  # the oldest record that retention kept
        self.committed = None

    def poll(self, offset, count):
        """Up to ``count`` records from ``offset`` on. A consumer with auto.offset.reset=none
        gets an error for an offset that retention removed."""
        if offset < self.start:
            raise RuntimeError(f"offset {offset} is out of range (the log starts at {self.start})")
        first = next((i for i, o in enumerate(self.offsets) if o >= offset), len(self.offsets))
        return list(zip(self.offsets[first : first + count], self.texts[first : first + count]))


partition = Partition("news", 0, news)
print("records:", len(partition.texts), "first offsets:", partition.offsets[:6])

## 2. The consumer loop

Three rules make the loop correct:

1. **Put topic, partition and offset into the position**, here `news:0:<offset>`. fse does not check it, so the loop checks it when it seeks.
2. **Keep a map from replay row to offset.** A replay row counts the records read since the seek. Offsets have gaps, so a row number is no offset. (An endless consumer drops the entries before the last commit.)
3. **Commit the offset of the last durable record plus one**, never a row count.

In [ ]:
def batches(part, position, sink_rows, size, rows_to_offsets):
    """Read ``part`` from the stored position on; yield (texts, position) for every poll."""
    topic, number, offset = position.split(":")
    assert (topic, int(number)) == (part.topic, part.number), "the position is of another partition"
    offset = int(offset)
    while True:
        records = part.poll(offset, size)
        if not records:
            return
        rows_to_offsets.extend(o for o, _ in records)
        yield [t for _, t in records], f"{part.topic}:{part.number}:{records[0][0]}"
        offset = records[-1][0] + 1


def consume(part, store_path, *, size=70, stop_after=None):
    """Write the records of ``part`` into a store; return the offsets committed on the way."""
    rows_to_offsets, commits = [], []
    with fse.ShardSink(store_path, encoder, shard_rows=128, resume=True) as sink:
        found = sink.source_position
        start = found[0] if found else f"{part.topic}:{part.number}:{part.start}"
        skip = found[1] if found else 0
        for done, result in enumerate(
            encoder.encode_batches(
                batches(part, start, sink.first_row, size, rows_to_offsets), sink=sink, positions=True
            ),
            1,
        ):
            if result.durable_rows:  # the last durable record, plus one
                part.committed = rows_to_offsets[result.durable_rows - 1] + 1
                commits.append(part.committed)
            if stop_after and done == stop_after:
                return commits  # a clean stop: leaving the with block closes the sink in order
        # the iteration ended and the sink flushed: everything read is durable now
        if rows_to_offsets:  # a new, empty partition has nothing to commit
            part.committed = rows_to_offsets[skip + sink.durable_rows - 1] + 1
            commits.append(part.committed)
    return commits

## 3. Crash and restart

The first run stops after four polls. The second run finds the position in the store, reads again from there, and finishes.

In [ ]:
store = work / "news-0"
first = consume(partition, store, stop_after=4)
with fse.VectorStore.open(store) as saved:
    rows_saved, (position, skip) = len(saved), saved.source_position
print({"saved_rows": rows_saved, "seek_to": position, "drop_first": skip})
second = consume(partition, store)
print("last commit of the second run:", second[-1], "- offset after the last record:", partition.offsets[-1] + 1)

The result equals a run that never stopped, **bit for bit**, with no record missing and none stored twice.

In [ ]:
with fse.VectorStore.open(store) as saved:
    saved.verify()
    stored = saved.get_vectors()
assert len(stored) == len(news)
assert stored.view(np.uint32).tobytes() == encoder.encode(news, workers=1).view(np.uint32).tobytes()
print("stored rows:", len(stored), "- every vector equals an uninterrupted run")

## 4. A store for each partition and day

A store belongs to **one partition**. For a long-running topic start a **new store each day** (or each hour): `close()` the sink at a batch boundary and open the next one.
After a restart continue in the newest store. A crash between two stores appends to the older one, because its position is the newest.
`VectorStore.open_view` reads the stores of a day, or of all partitions, as one.

In [ ]:
second_partition = Partition("news", 1, news[:300])
consume(second_partition, work / "news-1")
with fse.VectorStore.open_view([work / "news-0", work / "news-1"]) as view:
    print({"rows": len(view), "stores": [m["rows"] for m in view.members]})
    print("positions:", [m["position"] for m in view.members])

## 5. If the log lost the record

Retention may delete the record at the stored position. Then the restart **must fail loudly** instead of reading from the start or the end.
Set `auto.offset.reset=none` on the consumer: it [throws an exception to the consumer](https://kafka.apache.org/41/configuration/consumer-configs/) when the offset no longer exists. The simulated partition does the same.

In [ ]:
lost = Partition("news", 2, news[:200])
consume(lost, work / "news-2", stop_after=2)
lost.start = lost.offsets[150]  # retention removed the first records
try:
    consume(lost, work / "news-2")
except RuntimeError as error:
    print("refused:", error)

## Before using your own topic

- Use `enable.auto.commit=false` and commit yourself, after `durable_rows` moved.
- Open the sink and seek in the callback that assigns a partition. Flush and close the sink in the callback that revokes it (`ConsumerRebalanceListener` in the Java client, `on_assign` and `on_revoke` in `confluent-kafka`).
- A compacted or rewritten log does not return the same records, so it does not resume.
- A live source that cannot be read again stays with `encode_batches` and at-least-once delivery, or needs a buffer that can be read again in front of it.
- A position of up to 64 KiB is allowed; keep it short.
- The store guarantee holds on a supported local filesystem, not on SMB or NFS.

## What next?

- [Kafka and Qdrant](04-kafka-and-qdrant.ipynb).
- [fse API and design](../docs/architecture/store.md#stream-positions)
- [confluent-kafka consumer guide](https://docs.confluent.io/kafka-clients/python/current/overview.html#kafka-consumer)

Back to the [recipe index](README.md) or the [main README](../README.md).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [ ]:
recipe.close()